# Task 4 — NLP Sentiment Classification on Customer Reviews (TF-IDF)

**Internship:** Machine Learning & Data Science  
**Task:** 4 — Sentiment classification  
**Objective:** Build a text-classification pipeline that predicts whether a review is positive or negative.

This notebook uses the UCI Sentiment Labelled Sentences dataset. It contains short reviews from different domains, including products, movies, and restaurants. The workflow covers basic text cleaning, TF-IDF features, model comparison, evaluation, and a small helper for testing new text.

> Run the notebook from top to bottom. The results shown in the output cells are generated by your run; no metric values are hard-coded here.


## 1. Imports and setup

In [ ]:
import io
import re
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)

print("Libraries loaded.")


Libraries loaded.


## 2. Load the review data

The UCI archive contains three tab-separated files. Each row has review text and a binary label (`1` = positive, `0` = negative). We combine the three files so the model sees reviews from multiple domains.

If the download fails, check your internet connection and rerun this cell.


In [3]:
DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"

with urllib.request.urlopen(DATA_URL, timeout=30) as response:
    archive_bytes = response.read()

reviews = []
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for filename in archive.namelist():
        if filename.endswith("_labelled.txt"):
            with archive.open(filename) as file:
                frame = pd.read_csv(
                    file,
                    sep="\t",
                    header=None,
                    names=["review", "sentiment"],
                    encoding="utf-8",
                )
                reviews.append(frame)

df = pd.concat(reviews, ignore_index=True)
df["review"] = df["review"].astype(str).str.strip()
df["sentiment"] = pd.to_numeric(df["sentiment"], errors="coerce")
df = df.dropna(subset=["review", "sentiment"])
df = df[df["review"].ne("")]
df["sentiment"] = df["sentiment"].astype(int)
df = df[df["sentiment"].isin([0, 1])].drop_duplicates(subset=["review", "sentiment"])

label_names = {0: "negative", 1: "positive"}
df["sentiment_name"] = df["sentiment"].map(label_names)

print("Dataset shape:", df.shape)
display(df.head())
print("\nClass counts:")
display(df["sentiment_name"].value_counts())


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa2 in position 37: invalid start byte

## 3. Quick data checks

In [ ]:
print("Missing values:")
display(df[["review", "sentiment"]].isna().sum())

print("Duplicate review rows:", df.duplicated(subset=["review", "sentiment"]).sum())
print("\nClass balance:")
class_counts = df["sentiment_name"].value_counts().reindex(["negative", "positive"])
ax = class_counts.plot(kind="bar", rot=0, figsize=(6, 4))
ax.set_title("Number of reviews by sentiment")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Review count")
plt.tight_layout()
plt.show()


## 4. Text cleaning

For this baseline, use a small regex-based cleaner to lowercase text and remove HTML fragments, URLs, punctuation, and extra whitespace. TF-IDF handles tokenization and stop-word filtering. Keeping this cleaning step simple makes the pipeline easy to inspect and reproduce.


In [ ]:
def clean_review(text):
    # Convert text to a consistent format before vectorization.
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)          # Remove HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # Remove links
    text = re.sub(r"[^a-z0-9\s']", " ", text)    # Keep words and apostrophes
    text = re.sub(r"\s+", " ", text).strip()     # Collapse repeated spaces
    return text

df["clean_review"] = df["review"].apply(clean_review)
df = df[df["clean_review"].ne("")].copy()

display(df[["review", "clean_review", "sentiment_name"]].head(8))


## 5. Train/test split

The test set is held aside until final evaluation. Stratification keeps the positive/negative ratio similar in both splits. The vectorizer is placed inside each model pipeline so vocabulary and IDF weights are learned from the training split only.


In [ ]:
X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training reviews: {len(X_train):,}")
print(f"Test reviews:     {len(X_test):,}")


## 6. Build and compare models

In [ ]:
models = {
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
        )),
        ("classifier", MultinomialNB()),
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
}

results = []
fitted_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    score = accuracy_score(y_test, predictions)

    fitted_models[model_name] = model
    results.append({
        "Model": model_name,
        "Accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df.style.format({"Accuracy": "{:.3f}"}))


## 7. Evaluate the selected model

In [ ]:
# Use the better test accuracy as a simple baseline selection rule.
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
test_predictions = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["negative", "positive"],
    digits=3,
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=[0, 1],
    display_labels=["negative", "positive"],
    cmap="Blues",
    values_format="d",
)
plt.title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 8. Inspect model errors

Reading incorrect predictions can help identify short, ambiguous, or context-dependent reviews. This is a small qualitative check, not a replacement for a larger evaluation.


In [ ]:
error_review = pd.DataFrame({
    "review": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(test_predictions),
})
error_review = error_review[error_review["actual"] != error_review["predicted"]].copy()
error_review["actual_label"] = error_review["actual"].map(label_names)
error_review["predicted_label"] = error_review["predicted"].map(label_names)

display(error_review[["review", "actual_label", "predicted_label"]].head(15))
print("Misclassified reviews:", len(error_review))


## 9. Try a new review

In [ ]:
def predict_sentiment(review_text, model=best_model):
    cleaned = clean_review(review_text)
    if not cleaned:
        raise ValueError("Please enter a review containing some text.")

    prediction = int(model.predict([cleaned])[0])
    confidence = None

    # Both models used here support predict_proba.
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba([cleaned])[0]
        confidence = float(probabilities[prediction])

    result = {
        "review": review_text,
        "predicted_sentiment": label_names[prediction],
    }
    if confidence is not None:
        result["model_probability_for_prediction"] = round(confidence, 3)
    return result

# Change this example to test a review of your own.
example_review = "The product works well and I am happy with the purchase."
predict_sentiment(example_review)


## 10. Conclusion and limitations

After running the notebook, record the measured test accuracy from the model comparison table and note which model performed better on this split.

### Notes
- Accuracy depends on the train/test split and the domains represented in the dataset.
- A model trained on short benchmark reviews may not generalize to every product, language, or review platform.
- The probability returned by the classifier is not a guarantee that the prediction is correct.
- For a production use case, evaluate on fresh data, inspect class-specific precision/recall, and monitor model performance over time.

### Reproducibility
The split uses `random_state=42`. Re-running with the same dataset and compatible library versions should produce reproducible results.
